# 📊 Enterprise Retail Sales — Exploratory Data Analysis

Comprehensive EDA of the Superstore Sales Dataset including data cleaning, trend analysis, customer segmentation, advanced analytics (RFM, association rules, anomaly detection), and sales forecasting.

**Dataset**: 9,994 US retail transactions (2014–2017) across Furniture, Office Supplies, and Technology categories.

## 1. Import Required Libraries

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Add project root to path for src imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_loader import load_and_clean_data
from src.feature_engine import add_temporal_features, compute_rfm, compute_association_rules, detect_anomalies
from src import analysis

# Plot defaults
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100
sns.set_theme(style="whitegrid", palette="muted")

print("Libraries loaded successfully ✓")

## 2. Load the Retail Sales Dataset

Load the raw CSV via the `data_loader` pipeline which handles ingestion, Pandera schema validation, and cleaning in a single call.

In [ ]:
RAW_PATH = os.path.join(PROJECT_ROOT, "data", "01_raw", "superstore.csv")
df = load_and_clean_data(RAW_PATH)
df = add_temporal_features(df)

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 3. Data Cleaning and Preprocessing

The `load_and_clean_data()` pipeline already handled:
- Missing value removal
- Duplicate elimination
- Datetime conversion (`Order Date`, `Ship Date`)
- Smart-quote normalisation in Product Names
- Pandera schema validation

Let's verify the clean state:

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDate range: {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"\nData types:")
df.dtypes

## 4. Exploratory Data Analysis — Sales Overview

In [ ]:
print(f"Shape: {df.shape}")
print(f"Unique Orders: {df['Order ID'].nunique():,}")
print(f"Unique Customers: {df['Customer ID'].nunique():,}")
print(f"Unique Products: {df['Product Name'].nunique():,}")
print(f"\nTotal Sales:  ${df['Sales'].sum():,.2f}")
print(f"Total Profit: ${df['Profit'].sum():,.2f}")
print(f"Profit Margin: {df['Profit'].sum() / df['Sales'].sum() * 100:.1f}%")
print()
df[["Sales", "Profit", "Quantity", "Discount"]].describe().round(2)

## 5. Sales Trends Over Time

Monthly aggregation reveals seasonal patterns and long-term growth trajectories.

In [ ]:
monthly = analysis.monthly_sales_trend(df)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly["YearMonth"], monthly["Sales"], marker="o", linewidth=2, label="Sales")
ax.plot(monthly["YearMonth"], monthly["Profit"], marker="s", linewidth=2, linestyle="--", label="Profit")
ax.axhline(0, color="red", linewidth=0.8, linestyle=":")
ax.set_title("Monthly Sales & Profit Trend (2014–2017)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Amount ($)")
ax.legend()
# Show every 6th tick to avoid crowding
ax.set_xticks(ax.get_xticks()[::6])
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "visuals", "monthly_trend.png"), dpi=150, bbox_inches="tight")
plt.show()
print("↑ Clear seasonal peaks in November–December each year (holiday effect).")

## 6. Top Performing Products and Categories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Category sales
cat = analysis.category_sales(df)
axes[0].barh(cat["Category"], cat["Sales"], color=["#4e79a7", "#f28e2b", "#e15759"])
axes[0].set_title("Sales by Category", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Revenue ($)")
for i, v in enumerate(cat["Sales"]):
    axes[0].text(v + 5000, i, f"${v:,.0f}", va="center", fontsize=10)

# Top 10 sub-categories
top = analysis.top_subcategories(df, n=10)
colors = sns.color_palette("viridis", n_colors=10)
axes[1].barh(top["Sub-Category"], top["Sales"], color=colors)
axes[1].set_title("Top 10 Sub-Categories by Revenue", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Revenue ($)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "visuals", "category_performance.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Customer Purchase Patterns

Analyze customer segments, purchase frequency, and average order value.

In [ ]:
# Segment analysis
seg = analysis.segment_analysis(df)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sales by segment
axes[0].pie(seg["Sales"], labels=seg["Segment"], autopct="%1.1f%%", startangle=90,
            colors=["#4e79a7", "#f28e2b", "#e15759"])
axes[0].set_title("Sales Distribution\nby Segment", fontweight="bold")

# Regional performance
reg = analysis.region_sales(df)
axes[1].bar(reg["Region"], reg["Sales"], color=["#59a14f", "#4e79a7", "#f28e2b", "#e15759"])
axes[1].set_title("Sales by Region", fontweight="bold")
axes[1].set_ylabel("Revenue ($)")
for i, v in enumerate(reg["Sales"]):
    axes[1].text(i, v + 5000, f"${v:,.0f}", ha="center", fontsize=9)

# Customer purchase frequency distribution
cust_freq = df.groupby("Customer ID")["Order ID"].nunique()
axes[2].hist(cust_freq, bins=20, color="#76b7b2", edgecolor="white")
axes[2].set_title("Purchase Frequency\nDistribution", fontweight="bold")
axes[2].set_xlabel("Number of Orders")
axes[2].set_ylabel("Customers")
axes[2].axvline(cust_freq.mean(), color="red", linestyle="--", label=f"Mean: {cust_freq.mean():.1f}")
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "visuals", "customer_segments.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Avg orders per customer: {cust_freq.mean():.1f}")
print(f"Avg order value: ${df.groupby('Order ID')['Sales'].sum().mean():,.2f}")

## 8. Revenue and Profit Analysis

Investigate the critical relationship between Sales and Profit — high revenue doesn't always mean healthy margins.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: Profit vs Sales with Category hue
for cat_name, grp in df.groupby("Category"):
    axes[0].scatter(grp["Sales"], grp["Profit"], alpha=0.4, label=cat_name, s=20)
axes[0].axhline(0, color="red", linestyle="--", linewidth=1, label="Break-even")
axes[0].set_title("Profit vs Sales by Category", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Sales ($)")
axes[0].set_ylabel("Profit ($)")
axes[0].legend()

# Profit margin by category
margin = analysis.profit_margin_by_category(df)
bar_colors = ["#59a14f" if m > 0 else "#e15759" for m in margin["Profit_Margin_Pct"]]
axes[1].bar(margin["Category"], margin["Profit_Margin_Pct"], color=bar_colors)
axes[1].set_title("Profit Margin by Category", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Profit Margin (%)")
axes[1].axhline(0, color="gray", linewidth=0.5)
for i, v in enumerate(margin["Profit_Margin_Pct"]):
    axes[1].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "visuals", "profit_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()

# Discount impact
disc = analysis.discount_impact(df)
print("\nImpact of Discount on Average Profit:")
print(disc.to_string(index=False))

## 9. Sales Forecasting with Moving Averages

Compute Simple Moving Average (SMA) and Exponential Moving Average (EMA) on monthly sales to identify short-term momentum.

In [ ]:
monthly_fc = monthly.copy()
monthly_fc["SMA_3"] = monthly_fc["Sales"].rolling(window=3).mean()
monthly_fc["SMA_6"] = monthly_fc["Sales"].rolling(window=6).mean()
monthly_fc["EMA_3"] = monthly_fc["Sales"].ewm(span=3, adjust=False).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly_fc["YearMonth"], monthly_fc["Sales"], marker="o", linewidth=1.5, alpha=0.6, label="Actual Sales")
ax.plot(monthly_fc["YearMonth"], monthly_fc["SMA_3"], linewidth=2, label="3-Month SMA", color="#e15759")
ax.plot(monthly_fc["YearMonth"], monthly_fc["SMA_6"], linewidth=2, label="6-Month SMA", color="#4e79a7")
ax.plot(monthly_fc["YearMonth"], monthly_fc["EMA_3"], linewidth=2, linestyle="--", label="3-Month EMA", color="#59a14f")
ax.set_title("Sales Forecast: Moving Averages", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($)")
ax.legend()
ax.set_xticks(ax.get_xticks()[::6])
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "visuals", "moving_averages.png"), dpi=150, bbox_inches="tight")
plt.show()

## 10. Key Sales Metrics Dashboard

A dashboard-style summary with multiple subplots showing total sales, average order value, regional breakdown, and month-over-month growth.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Executive Sales Dashboard", fontsize=16, fontweight="bold", y=1.01)

# 1. Total Sales & Profit KPI text
axes[0, 0].axis("off")
kpi_text = (
    f"Total Sales\n${df['Sales'].sum():,.0f}\n\n"
    f"Total Profit\n${df['Profit'].sum():,.0f}\n\n"
    f"Margin: {df['Profit'].sum()/df['Sales'].sum()*100:.1f}%"
)
axes[0, 0].text(0.5, 0.5, kpi_text, ha="center", va="center", fontsize=14, fontweight="bold",
                transform=axes[0, 0].transAxes,
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#e6f2ff", alpha=0.8))
axes[0, 0].set_title("Key Performance Indicators", fontweight="bold")

# 2. Sales by Category
cat = analysis.category_sales(df)
axes[0, 1].bar(cat["Category"], cat["Sales"], color=["#4e79a7", "#f28e2b", "#e15759"])
axes[0, 1].set_title("Sales by Category", fontweight="bold")
axes[0, 1].set_ylabel("Revenue ($)")

# 3. Sales by Region
reg = analysis.region_sales(df)
axes[0, 2].bar(reg["Region"], reg["Sales"], color=["#59a14f", "#4e79a7", "#f28e2b", "#e15759"])
axes[0, 2].set_title("Sales by Region", fontweight="bold")
axes[0, 2].set_ylabel("Revenue ($)")

# 4. Monthly trend
axes[1, 0].plot(monthly["YearMonth"], monthly["Sales"], marker=".", linewidth=1.5, color="#4e79a7")
axes[1, 0].set_title("Monthly Revenue Trend", fontweight="bold")
axes[1, 0].set_xticks(axes[1, 0].get_xticks()[::12])
axes[1, 0].tick_params(axis="x", rotation=45)

# 5. MoM Growth Rate
monthly_growth = monthly.copy()
monthly_growth["MoM_Growth"] = monthly_growth["Sales"].pct_change() * 100
colors = ["#59a14f" if g >= 0 else "#e15759" for g in monthly_growth["MoM_Growth"]]
axes[1, 1].bar(range(len(monthly_growth)), monthly_growth["MoM_Growth"], color=colors, width=0.8)
axes[1, 1].axhline(0, color="black", linewidth=0.5)
axes[1, 1].set_title("Month-over-Month Growth (%)", fontweight="bold")
axes[1, 1].set_ylabel("Growth (%)")

# 6. Sales distribution
axes[1, 2].hist(df["Sales"], bins=50, color="#76b7b2", edgecolor="white", log=True)
axes[1, 2].set_title("Sales Distribution (log scale)", fontweight="bold")
axes[1, 2].set_xlabel("Order Sales ($)")
axes[1, 2].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "visuals", "executive_dashboard.png"), dpi=150, bbox_inches="tight")
plt.show()

---

## 11. Advanced Analytics — RFM Customer Segmentation

Segment the customer base by **Recency** (days since last purchase), **Frequency** (number of orders), and **Monetary** value (total spend). Customers are scored 1–5 per dimension and mapped to behavioral segments.

In [ ]:
rfm = compute_rfm(df)
print(f"RFM computed for {len(rfm):,} unique customers\n")
print("Segment distribution:")
print(rfm["Segment"].value_counts())

fig, ax = plt.subplots(figsize=(10, 5))
seg_counts = rfm["Segment"].value_counts()
ax.barh(seg_counts.index, seg_counts.values, color=sns.color_palette("viridis", len(seg_counts)))
ax.set_title("RFM Customer Segmentation", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of Customers")
for i, v in enumerate(seg_counts.values):
    ax.text(v + 3, i, str(v), va="center")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "visuals", "rfm_segments.png"), dpi=150, bbox_inches="tight")
plt.show()

## 12. Advanced Analytics — Association Rule Mining

Discover product co-purchase patterns using the FP-Growth algorithm at the Sub-Category level.

In [ ]:
rules = compute_association_rules(df)
print(f"Found {len(rules)} association rules\n")

if not rules.empty:
    display_rules = rules.head(15).copy()
    display_rules["antecedents"] = display_rules["antecedents"].apply(lambda x: ", ".join(list(x)))
    display_rules["consequents"] = display_rules["consequents"].apply(lambda x: ", ".join(list(x)))
    print("Top 15 Association Rules by Lift:")
    display_rules[["antecedents", "consequents", "support", "confidence", "lift"]].round(3)

## 13. Advanced Analytics — Anomaly Detection (Isolation Forest)

Flag suspicious transactions that deviate significantly from normal Sales/Profit/Quantity/Discount patterns.

In [ ]:
df_anom = detect_anomalies(df)
n_anomalies = df_anom["Is_Anomaly"].sum()
print(f"Anomalies detected: {n_anomalies} / {len(df_anom)} ({n_anomalies/len(df_anom)*100:.1f}%)\n")

fig, ax = plt.subplots(figsize=(12, 6))
normal = df_anom[~df_anom["Is_Anomaly"]]
anomalous = df_anom[df_anom["Is_Anomaly"]]
ax.scatter(normal["Sales"], normal["Profit"], alpha=0.3, s=15, color="steelblue", label="Normal")
ax.scatter(anomalous["Sales"], anomalous["Profit"], alpha=0.7, s=30, color="red", marker="x", label="Anomaly")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Anomaly Detection — Isolation Forest", fontsize=13, fontweight="bold")
ax.set_xlabel("Sales ($)")
ax.set_ylabel("Profit ($)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "visuals", "anomaly_detection.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nTop flagged anomalous transactions:")
anomalous[["Order ID", "Product Name", "Category", "Sales", "Profit", "Discount"]].sort_values("Profit").head(10)

---

## Summary of Key Business Insights

1. **Technology Dominance** — Technology generates the highest aggregate revenue, serving as the primary revenue engine for the corporation.

2. **West Region Superiority** — The West consistently outpaces all other territories in gross revenue.

3. **December Peak** — Massive seasonal spike in Q4/December driven by holiday purchasing patterns.

4. **High Sales / Negative Profit Anomaly** — A significant cluster of transactions shows high revenue but negative margins — a clear indicator of catastrophic discounting policies eroding profitability.

5. **RFM Segmentation** — Customer base is clearly stratified into actionable segments (Champions, Loyal, At-Risk) enabling targeted marketing.

6. **Product Co-purchase Patterns** — Association rules reveal natural product bundling opportunities for cross-sell optimization.